# **Problem Statement**

## **Business Context**

Workplace safety in hazardous environments like construction sites and industrial plants is crucial to prevent accidents and injuries. One of the most important safety measures is ensuring workers wear safety helmets, which protect against head injuries from falling objects and machinery. Non-compliance with helmet regulations increases the risk of serious injuries or fatalities, making effective monitoring essential, especially in large-scale operations where manual oversight is prone to errors and inefficiency.

To overcome these challenges, SafeGuard Corp plans to develop an automated image analysis system capable of detecting whether workers are wearing safety helmets. This system will improve safety enforcement, ensuring compliance and reducing the risk of head injuries. By automating helmet monitoring, SafeGuard aims to enhance efficiency, scalability, and accuracy, ultimately fostering a safer work environment while minimizing human error in safety oversight.

## **Objective**

As a data scientist at SafeGuard Corp, you are tasked with developing an image classification model that classifies images into one of two categories:
- **With Helmet:** Workers wearing safety helmets.
- **Without Helmet:** Workers not wearing safety helmets.

## **Data Description**

The dataset consists of **4125 images**, divided into two categories:

- **With Helmet:** 3161 images showing workers wearing helmets.
- **Without Helmet:** 964 images showing workers not wearing helmets.

**Dataset Characteristics:**
- **Variations in Conditions:** Images include diverse environments such as construction sites, factories, and industrial settings, with variations in lighting, angles, and worker postures to simulate real-world conditions.
- **Worker Activities:** Workers are depicted in different actions such as standing, using tools, or moving, ensuring robust model learning for various scenarios.

# **Installing and Importing the Necessary Libraries**

Install the dependencies once from the `requirements.txt` that ships with this notebook, then restart the kernel:

```
pip install -r requirements.txt
```

The original notebook pinned exact versions (for example `numpy==2.0.2`) that only suit one Colab image. The requirements file uses minimum versions instead.

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import tensorflow as tf
import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Flatten, Conv2D, MaxPooling2D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.applications.vgg16 import VGG16

from sklearn.model_selection import train_test_split
from sklearn.utils import class_weight
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, precision_recall_fscore_support, accuracy_score

import warnings
warnings.filterwarnings('ignore')

## Configuration

All settings live in this one cell. The three `HN_*` environment variables exist so the notebook can be smoke-tested quickly (fewer epochs, no weight download); leave them unset for a real run.

In [ ]:
# --- Where the data lives --------------------------------------------------
# Folder containing HN_images.npy and HN_labels.csv. Defaults to the current folder.
DATA_DIR = os.environ.get("HN_DATA_DIR", ".")

# Optional: on Google Colab, read the files from Drive instead.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = "/content/drive/MyDrive/Colab Notebooks"
except ImportError:
    pass

IMAGES_PATH = os.path.join(DATA_DIR, "HN_images.npy")
LABELS_PATH = os.path.join(DATA_DIR, "HN_labels.csv")

# --- Training settings -----------------------------------------------------
SEED = 812
SPLIT_SEED = 42
EPOCHS = int(os.environ.get("HN_EPOCHS", 20))
BATCH_SIZE = 32
LEARNING_RATE = 0.001
IMG_SHAPE = (200, 200, 3)

# 'imagenet' downloads the pre-trained VGG16 weights; set HN_VGG_WEIGHTS=none to skip (smoke tests only).
VGG_WEIGHTS = os.environ.get("HN_VGG_WEIGHTS", "imagenet")
if VGG_WEIGHTS.lower() == "none":
    VGG_WEIGHTS = None

CLASS_NAMES = ["Without Helmet", "With Helmet"]   # label 0, label 1

In [ ]:
print("TensorFlow:", tf.__version__)
print("GPUs available:", len(tf.config.list_physical_devices('GPU')))

In [ ]:
# Make runs repeatable: Python, NumPy and TensorFlow seeds, plus deterministic ops
random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()

# **Data Overview**

## Loading the data

In [ ]:
images = np.load(IMAGES_PATH)
labels = pd.read_csv(LABELS_PATH)

# Sanity checks: one label per image, binary labels, expected image size
assert images.shape[0] == len(labels), f"{images.shape[0]} images but {len(labels)} labels"
assert images.shape[1:] == IMG_SHAPE, f"unexpected image shape {images.shape[1:]}"
assert set(labels['label'].unique()) <= {0, 1}, "labels must be 0 (Without Helmet) or 1 (With Helmet)"
y = labels['label'].to_numpy()

# **Exploratory Data Analysis**

### Plot random images from each of the classes and print their corresponding labels.

In [ ]:
print("Images:", images.shape, images.dtype)
print("Labels:", labels.shape)

### Visualizing Random Images with Labels

In [ ]:
plt.figure(figsize=(10, 5))
for i, label_val in enumerate(sorted(np.unique(y))):
    random_index = np.random.choice(np.where(y == label_val)[0])
    plt.subplot(1, 2, i + 1)
    plt.imshow(images[random_index])
    plt.title(f"Label {label_val}: {CLASS_NAMES[label_val]}")
    plt.axis('off')
    print(f"Image index: {random_index}, Label: {label_val}")
plt.tight_layout()
plt.show()

## Checking for class imbalance

In [ ]:
counts = pd.Series(y).map(dict(enumerate(CLASS_NAMES))).value_counts()
print(counts)
print((counts / counts.sum()).round(3))
sns.countplot(y=pd.Series(y).map(dict(enumerate(CLASS_NAMES))))
plt.show()

The data is imbalanced (roughly three quarters of the images show a helmet). Accuracy alone would flatter a model that favours the majority class, so the models below are trained with **class weights** and judged on per-class recall and precision.

# **Data Preprocessing**

### Splitting the dataset

80% train, 10% validation, 10% test, stratified on the label so each split keeps the same class mix. (The labels file is sorted by class, so a stratified, shuffled split is required.)

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(images, y, test_size=0.2, random_state=SPLIT_SEED, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=SPLIT_SEED, stratify=y_temp)

In [ ]:
for name, Xs, ys in [("train", X_train, y_train), ("val", X_val, y_val), ("test", X_test, y_test)]:
    print(f"{name:5s} {Xs.shape}  share of 'With Helmet': {ys.mean():.3f}")

### Data Normalization

In [ ]:
# Scale pixels to [0, 1]. Nothing is fitted on the data, so there is no leakage between splits.
X_train_normalized = X_train.astype('float32') / 255.0
X_val_normalized = X_val.astype('float32') / 255.0
X_test_normalized = X_test.astype('float32') / 255.0

# Free the memory held by arrays that are no longer needed (X_val / X_test stay for plotting)
del images, X_temp, X_train

# **Model Building**

## Model Evaluation Criterion

The business goal is to catch workers who are **not** wearing a helmet. So the key numbers are, for the *Without Helmet* class (label 0):

- **Recall**: of the workers really without a helmet, how many did the model catch? A miss is a safety and insurance risk.
- **Precision**: of the workers flagged as without a helmet, how many really were? Low precision means false alarms.

Accuracy and macro F1 are reported for context. In a two-class problem, micro-averaged recall, precision and F1 all equal accuracy, so the original notebook's micro columns carried no extra information and are dropped.

## Utility Functions

In [ ]:
def predict_labels(model, X, threshold=0.5):
    """Predicted class (0/1) from the model's sigmoid output."""
    probs = model.predict(X, verbose=0).reshape(-1)
    return (probs > threshold).astype(int)


def classification_report_df(y_true, y_pred):
    """One-row DataFrame of the metrics used to compare models (class 0 = Without Helmet)."""
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, labels=[0, 1], zero_division=0)
    return pd.DataFrame({
        "Accuracy": accuracy_score(y_true, y_pred),
        "Recall (Without Helmet)": r[0],
        "Precision (Without Helmet)": p[0],
        "F1 (Without Helmet)": f[0],
        "Recall (With Helmet)": r[1],
        "Precision (With Helmet)": p[1],
        "F1 (With Helmet)": f[1],
        "F1 (Macro)": f.mean(),
    }, index=[0])


def plot_confusion_matrix(y_true, y_pred, title="Confusion Matrix"):
    """Row-normalised confusion matrix: each row shows how a true class was classified."""
    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay.from_predictions(
        y_true, y_pred, labels=[0, 1], normalize='true', cmap='Blues',
        values_format='.2f', display_labels=CLASS_NAMES, ax=ax)
    ax.set_title(title)
    plt.show()


def evaluate_model(model, name):
    """Predict once on train and validation, print metrics, plot confusion matrices.
    Returns (train_perf, valid_perf). The test set is deliberately NOT touched here."""
    train_pred = predict_labels(model, X_train_normalized)
    val_pred = predict_labels(model, X_val_normalized)
    train_perf = classification_report_df(y_train, train_pred)
    valid_perf = classification_report_df(y_val, val_pred)
    print(f"{name}: train performance"); print(train_perf.T.round(4)); print()
    print(f"{name}: validation performance"); print(valid_perf.T.round(4))
    plot_confusion_matrix(y_train, train_pred, f"{name}: train")
    plot_confusion_matrix(y_val, val_pred, f"{name}: validation")
    return train_perf, valid_perf


def plot_history(history, name):
    plt.plot(history.history['accuracy'])
    plt.plot(history.history['val_accuracy'])
    plt.title(f'{name}: accuracy')
    plt.ylabel('Accuracy'); plt.xlabel('Epoch')
    plt.legend(['Train', 'Validation'], loc='lower right')
    plt.show()


def show_predictions(model, indices):
    """Show validation images with the model's prediction and the true label."""
    indices = [i for i in indices if i < len(X_val)]
    fig, axes = plt.subplots(1, len(indices), figsize=(3 * len(indices), 3.4))
    for ax, idx in zip(np.atleast_1d(axes), indices):
        pred = int(model.predict(X_val_normalized[idx:idx + 1], verbose=0)[0][0] > 0.5)
        ax.imshow(X_val[idx]); ax.axis('off')
        ax.set_title(f"Pred: {CLASS_NAMES[pred]}\nTrue: {CLASS_NAMES[y_val[idx]]}", fontsize=9)
    plt.show()


def make_vgg_base():
    """Fresh, frozen VGG16 convolutional base (so models do not share state)."""
    base = VGG16(weights=VGG_WEIGHTS, include_top=False, input_shape=IMG_SHAPE)
    base.trainable = False
    return base

### Handling Class Imbalance with Class Weights

In [ ]:
class_weights = class_weight.compute_class_weight(class_weight='balanced', classes=np.array([0, 1]), y=y_train)
class_weights_dict = {i: float(w) for i, w in enumerate(class_weights)}
print("Class weights:", class_weights_dict)

# Model 1: A basic CNN

In [ ]:
model_1 = Sequential([
    Conv2D(16, (3, 3), activation='relu', padding="same", input_shape=IMG_SHAPE),
    MaxPooling2D((4, 4), padding='same'),
    Conv2D(32, (3, 3), activation='relu', padding="same"),
    MaxPooling2D((2, 2), padding='same'),
    Conv2D(64, (3, 3), activation='relu', padding="same"),
    Flatten(),
    Dense(4, activation='relu'),
    Dense(1, activation='sigmoid'),
])
model_1.compile(optimizer=Adam(learning_rate=LEARNING_RATE), loss='binary_crossentropy', metrics=["accuracy"])
model_1.summary()

In [ ]:
history_1 = model_1.fit(
    X_train_normalized, y_train,
    epochs=EPOCHS, batch_size=BATCH_SIZE, shuffle=True,
    validation_data=(X_val_normalized, y_val),
    class_weight=class_weights_dict, verbose=2)

In [ ]:
plot_history(history_1, 'Model 1')

In [ ]:
model_1_train_perf, model_1_valid_perf = evaluate_model(model_1, 'Model 1')

### Visualizing the predictions

In [ ]:
show_predictions(model_1, [12, 33])

# Model 2: Transfer learning with VGG-16 (base only)

VGG16 has 16 weight layers (13 convolutional, 3 fully connected). Here only the 13 convolutional layers are used, pre-trained on ImageNet and **frozen**; a single sigmoid unit is trained on top.

In [ ]:
model_2 = Sequential([make_vgg_base(), Flatten(), Dense(1, activation='sigmoid')])
model_2.compile(optimizer=Adam(learning_rate=LEARNING_RATE), loss='binary_crossentropy', metrics=["accuracy"])
model_2.summary()

In [ ]:
# No augmentation here; the generator only serves shuffled batches.
train_datagen = ImageDataGenerator()
history_2 = model_2.fit(
    train_datagen.flow(X_train_normalized, y_train, batch_size=BATCH_SIZE, seed=SEED, shuffle=True),
    epochs=EPOCHS, validation_data=(X_val_normalized, y_val),
    class_weight=class_weights_dict, verbose=2)

In [ ]:
plot_history(history_2, 'Model 2')

In [ ]:
model_2_train_perf, model_2_valid_perf = evaluate_model(model_2, 'Model 2')

### Visualizing the predictions

In [ ]:
show_predictions(model_2, [0, 1])

# Model 3: Transfer learning with VGG-16 (base + feed-forward layers)

Frozen VGG16 base, plus two trainable dense layers (128 and 64 units, 30% dropout between them) and a sigmoid output.

In [ ]:
model_3 = Sequential([
    make_vgg_base(), Flatten(),
    Dense(128, activation='relu'),
    Dropout(rate=0.3),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid'),
])
model_3.compile(optimizer=Adam(learning_rate=LEARNING_RATE), loss='binary_crossentropy', metrics=["accuracy"])
model_3.summary()

In [ ]:
history_3 = model_3.fit(
    train_datagen.flow(X_train_normalized, y_train, batch_size=BATCH_SIZE, seed=SEED, shuffle=True),
    epochs=EPOCHS, validation_data=(X_val_normalized, y_val),
    class_weight=class_weights_dict, verbose=2)

In [ ]:
plot_history(history_3, 'Model 3')

In [ ]:
model_3_train_perf, model_3_valid_perf = evaluate_model(model_3, 'Model 3')

### Visualizing the predictions

In [ ]:
show_predictions(model_3, [0, 1])

# Model 4: Transfer learning with VGG-16 (base + feed-forward layers + data augmentation)

- In most real-world projects it is hard to collect a large number of images. **Data augmentation** creates new training examples by transforming the existing ones.
- Techniques available: horizontal / vertical flip, height and width shift (0 to 1), rotation (0 to 180 degrees), shear (0 to 1) and zoom (0 to 1).
- Workers in this dataset are photographed upright, so the augmentation below is kept realistic: horizontal flips, small rotations, small shifts and zoom. The original notebook also used vertical flips and rotations up to 180 degrees, which produce upside-down workers that never occur at test time. If you want to reproduce that setting, change `rotation_range` to 180 and `vertical_flip` to True.

In [ ]:
model_4 = Sequential([
    make_vgg_base(), Flatten(),
    Dense(128, activation='relu'),
    Dropout(rate=0.3),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid'),
])
model_4.compile(optimizer=Adam(learning_rate=LEARNING_RATE), loss='binary_crossentropy', metrics=["accuracy"])
model_4.summary()

***Applying data augmentation*** (training images only; validation and test images are never augmented)

In [ ]:
augmenting_datagen = ImageDataGenerator(
    rotation_range=20,
    horizontal_flip=True,
    vertical_flip=False,
    fill_mode='nearest',
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.01,
    zoom_range=0.2,
)
history_4 = model_4.fit(
    augmenting_datagen.flow(X_train_normalized, y_train, batch_size=BATCH_SIZE, seed=SEED, shuffle=True),
    epochs=EPOCHS, validation_data=(X_val_normalized, y_val),
    class_weight=class_weights_dict, verbose=2)

In [ ]:
plot_history(history_4, 'Model 4')

In [ ]:
model_4_train_perf, model_4_valid_perf = evaluate_model(model_4, 'Model 4')

### Visualizing the predictions

In [ ]:
show_predictions(model_4, [0, 1])

# **Model Performance Comparison and Final Model Selection**

In [ ]:
model_names = ["Simple CNN", "VGG-16 (Base)", "VGG-16 (Base+FFNN)", "VGG-16 (Base+FFNN+Data Aug)"]
models = {n: m for n, m in zip(model_names, [model_1, model_2, model_3, model_4])}

models_train_comp_df = pd.concat([p.T for p in [model_1_train_perf, model_2_train_perf, model_3_train_perf, model_4_train_perf]], axis=1)
models_valid_comp_df = pd.concat([p.T for p in [model_1_valid_perf, model_2_valid_perf, model_3_valid_perf, model_4_valid_perf]], axis=1)
models_train_comp_df.columns = model_names
models_valid_comp_df.columns = model_names

print("Training performance"); display(models_train_comp_df.round(4))
print("Validation performance"); display(models_valid_comp_df.round(4))
print("Train minus validation (a large gap means overfitting)"); display((models_train_comp_df - models_valid_comp_df).round(4))

### Choosing the final model

The model is chosen **on the validation set only**, so the test set stays an honest, one-time estimate. Because missing a worker without a helmet is costlier than a false alarm, the models are ranked by the **F2 score of the *Without Helmet* class**, which weights recall twice as heavily as precision. (A model that flags everyone as helmetless would have perfect recall, but its precision, and so its F2, would be poor.)

In [ ]:
def f_beta(precision, recall, beta=2.0):
    denom = beta**2 * precision + recall
    return 0.0 if denom == 0 else (1 + beta**2) * precision * recall / denom

selection = models_valid_comp_df.loc[["Recall (Without Helmet)", "Precision (Without Helmet)"]].T.copy()
selection["F2 (Without Helmet)"] = [f_beta(p, r) for p, r in zip(selection["Precision (Without Helmet)"], selection["Recall (Without Helmet)"])]
display(selection.round(4))

best_name = selection["F2 (Without Helmet)"].idxmax()
best_model = models[best_name]
print("Selected model:", best_name)

## Test Performance

The selected model is evaluated on the test set once.

In [ ]:
test_pred = predict_labels(best_model, X_test_normalized)
test_perf = classification_report_df(y_test, test_pred)
print(f"{best_name}: test performance"); display(test_perf.T.round(4))
plot_confusion_matrix(y_test, test_pred, f"{best_name}: test")

In [ ]:
tn, fp, fn, tp = confusion_matrix(y_test, test_pred, labels=[0, 1]).ravel()
print(f"Of {tn + fp} test workers without a helmet, {tn} were caught and {fp} were missed ({fp / (tn + fp):.1%} missed).")
print(f"Of {fn + tp} test workers with a helmet, {fn} were wrongly flagged as helmetless ({fn / (fn + tp):.1%} false alarms).")

# **Actionable Insights & Recommendations**

The numbers behind these points are printed in the comparison tables and the test-set summary above, and they change if the notebook is re-run with different data or settings. Fill in the figures from your own run.

**How the models were judged.** The goal is to catch workers who are not wearing helmets, so the models are compared on recall and precision for the *Without Helmet* class (via the F2 score), not on accuracy alone. Compare the train-versus-validation gap table to see which models overfit.

**Recommendations**

- **Keep a human in the loop.** Any *Without Helmet* detection should be reviewed by a person before action is taken. The false-alarm rate reported on the test set shows how often a helmeted worker would be wrongly flagged.
- **Monitor the right metrics in production.** Track recall for *Without Helmet* (missed non-compliance) and precision for *Without Helmet* (false alarms), plus the share of images in each class, to catch drift.
- **Collect more diverse data**, especially the lighting conditions, angles and postures the model gets wrong, and more *Without Helmet* examples (about a quarter of the data).
- **Consider next steps**: fine-tuning the last VGG16 block, using VGG16's own `preprocess_input` scaling instead of dividing by 255, trying a lighter pre-trained network such as MobileNetV2 for deployment on site cameras, and tuning the decision threshold (0.5 is the default) to trade recall against false alarms.